# 02-handle-ref — 버리지 않고 밖에 두기

무손실([`01`](../01-lossless-structure/run.ipynb))은 표현의 중복까지가
천장입니다. 더 줄이려면 뭔가를 빼야 하는데, 빼는 방법이 둘입니다.

| | 정보는 | 되돌리기 |
|---|---|---|
| [`03-summarize-llm`](../03-summarize-llm/) | 요약하며 **없앱니다** | 불가 |
| **`02-handle-ref`** | 밖에 두고 **핸들만** 남깁니다 | 꺼내면 원문 그대로 |

핸들 방식은 아무것도 잃지 않습니다. 원문은 저장소에 그대로 있습니다.
대신 **꺼낼 것을 골라야** 하고, 잘못 고르면 답을 못 합니다.

> **이 랩의 결론을 미리 말하면** — 같은 비용(k=1)으로 보존율이 100% 도 되고
> 43.8% 도 됩니다. 차이는 압축 알고리즘이 아니라 **고르는 방법**입니다.

## 1. kit 과 블록 모듈 불러오기

In [ ]:
import sys
from pathlib import Path

LAB = Path.cwd().resolve()
LABS = LAB.parents[0]                  # labs/<이 랩> -> labs
sys.path.insert(0, str(LABS))
sys.path.insert(0, str(LAB))           # 이 랩의 모듈(transforms, blocks 등)

# 노트북을 켜 둔 채로 저장소를 갱신하면 커널이 **예전 코드를 물고 있습니다.**
# 그러면 새로 생긴 함수가 없다는 에러(AttributeError)가 나는데, 원인이 코드가
# 아니라 커널이라 찾기가 어렵습니다. 그래서 이 셀을 돌릴 때마다 새로 읽습니다.
_stale = [m for m in list(sys.modules)
          if m == "kit" or m.startswith("kit.")
          or m in ("transforms", "blocks", "summarize", "compress")]
for _m in _stale:
    del sys.modules[_m]

from kit import VERSION, config as C, dataset, env, metrics, tokens as T
from kit.display import table, pct
from kit.runner import Run

# .env 는 labs/.env → 저장소 루트 .env → scripts/explore/.env 순으로 찾습니다.
env.load(verbose=True)

RUNS = LABS.parent / "runs"
print("kit", VERSION, "· 랩", LAB.name)
if _stale:
    print(f"모듈 {len(_stale)}개를 새로 읽었습니다 — 커널에 남아 있던 예전 코드를 지웠습니다")

import blocks as B
from compress import compress

## 2. 블록으로 쪼개고 핸들 붙이기

블록 경계를 어디로 잡느냐가 라우팅 난이도를 정합니다. 문서에 이미 절 표시가
있으면 그걸 씁니다 — 사람이 의미 단위로 나눠 둔 것이라 기계가 다시 나누는
것보다 낫습니다.

In [ ]:
cases = dataset.load("../data/sample-long")
c = cases[0]

bs = B.make_blocks(c.text, "auto")
table(
    ["핸들", "제목", "길이", "질문과의 겹침"],
    [[f"[[{b.handle}]]", b.title or "(제목 없음)", f"{len(b.body)}자",
      f"{B.score(c.question, b):.3f}"] for b in bs],
    align=["left", "left", "right", "right"],
    title=f"{c.id} 를 블록으로 — 질문: {c.question}",
    note="겹침이 가장 큰 블록을 펼칩니다. 정답 절은 "
         f"'{c.meta['answer_section']}' 입니다.",
)

## 3. 컨텍스트에 실제로 들어가는 것

펼친 블록은 원문 그대로, 나머지는 **다이제스트 한 줄**로 들어갑니다.
다이제스트가 필요한 이유는, 흔적이 없으면 모델이 그런 내용이 있었는지조차
모르고 꺼낼 판단도 못 하기 때문입니다. 이 흔적의 길이가 곧 오버헤드입니다.

In [ ]:
counter = T.make_counter({}, "gpt-5.4")

for k in (0, 1):
    out, meta = compress(c.text, question=c.question, expand_k=k, digest_chars=24)
    print(f"── expand_k={k} · {counter(out):,} 토큰 "
          f"(원문 {counter(c.text):,}) · 펼친 절 {meta['expanded_titles']} ──")
    print(out[:420] + ("…" if len(out) > 420 else ""))
    print()

## 4. 라우팅이 있고 없고 — 이 랩의 핵심

`route` 를 바꿔 같은 `k` 로 비교합니다.

| | 무엇 |
|---|---|
| `bigram` | 질문과 글자 2-gram 이 많이 겹치는 블록 |
| `first` | 문서 **앞에서부터** — 가장 흔한 절단 방식 |

한국어는 띄어쓰기로 자르면 조사 때문에 잘 안 맞습니다(`수수료율은` vs
`수수료`). 그래서 글자 2-gram 겹침을 씁니다.

In [ ]:
def sweep(route, ks, cases, digest_chars=24):
    rows = []
    for k in ks:
        recs, hit = [], 0
        for cc in cases:
            after, extra = compress(cc.text, question=cc.question or "",
                                    expand_k=k, route=route,
                                    digest_chars=digest_chars)
            if cc.meta.get("answer_section") in extra["expanded_titles"]:
                hit += 1
            recs.append(metrics.per_case(cc.id, cc.kind, cc.text, after,
                                         cc.must_include, counter, extra))
        mm = metrics.aggregate(recs, counter)
        rows.append([str(k), f'{mm["tokens_after"]:,}', pct(mm["saved"]),
                     pct(mm["survival_mean"]), pct(mm["survival_worst"]),
                     pct(hit / len(cases))])
    return rows


for route, label in [("bigram", "라우팅 있음"), ("first", "라우팅 없음 (앞에서 자르기)")]:
    table(
        ["펼침", "토큰", "절감", "평균 보존율", "최저 보존율", "정답 절 적중"],
        sweep(route, [0, 1, 2, 3, "all"], cases),
        align=["right"] * 6,
        title=f"{label} — route={route}",
    )
print(f"원문 {sum(counter(x.text) for x in cases):,} 토큰")

`first` 는 4개를 펼쳐도(=절감을 17.6% 까지 포기해도) 최저 보존율이 **0%**
입니다. 정답이 뒤쪽 절에 있는 문서는 아무리 앞을 남겨도 못 찾기 때문입니다.

> **토큰을 더 쓴다고 정확도가 오르지 않습니다.** 어디를 남기느냐가 정합니다.
> 코퍼스를 만들 때 정답 절의 위치를 문서마다 다르게 둔 이유입니다 — 항상
> 앞에 있으면 앞에서 자르기만으로도 통과해서 라우터를 평가할 수 없습니다.

## 5. 이 랩의 모든 조건 돌려보기

**조건 1개 = 파일 1개**입니다. `configs/` 를 훑으면 이 랩이 답할 수 있는
질문이 전부 나옵니다. 설정을 새로 추가해도 이 셀은 고칠 필요가 없습니다.

각 조건은 `runs/02-handle-ref/<설정이름>/<시각>/` 에 따로 기록됩니다. 나중에
"그때 무엇을 돌렸나" 를 설정 이름만 보고 알 수 있게 하려는 것입니다.

| 설정 | 무엇을 보려고 |
|---|---|
| `k1` | 기본 조건 — 질문에 맞는 절 1개 |
| `k0-digest-only` | 절감의 상한이자 보존율의 하한 |
| `k1-title-only` | 다이제스트를 제목만으로 줄이면 |
| `k1-no-router` | **대조군** — 라우팅 없이 앞에서 |
| `k1-short` | 짧은 산문에 쓰면 어떻게 되나 |

In [ ]:
def run_config(path):
    cfg = C.load(path)
    cs = dataset.load(cfg.dataset["path"], limit=cfg.dataset.get("limit"))
    cnt = T.make_counter(cfg.tokenizer, cfg.model)

    run = Run(cfg, RUNS)
    hit_known = hit_ok = 0
    for cc in cs:
        after, extra = compress(cc.text, question=cc.question or "", **cfg.params)
        want = cc.meta.get("answer_section")
        if want:
            hit_known += 1
            hit_ok += want in extra["expanded_titles"]
        run.add(metrics.per_case(cc.id, cc.kind, cc.text, after,
                                 cc.must_include, cnt, extra),
                before=cc.text, after=after)

    m = metrics.aggregate(run.records, cnt)
    m["dataset_name"] = Path(cfg.dataset["path"]).name
    m["expand_k"] = str(cfg.params.get("expand_k"))
    m["route"] = cfg.params.get("route", "bigram")
    m["digest_chars"] = cfg.params.get("digest_chars", 24)
    if hit_known:
        m["router_hit_rate"] = round(hit_ok / hit_known, 4)
    return cfg, m, run.finish(m, ["절감률은 **꺼내기 전** 기준입니다."])


results = []
for p in sorted(Path("configs").glob("*.yaml")):
    cfg, m, out = run_config(p)
    results.append((cfg.name, m, out))
    print(f'{cfg.name:18s} k={m["expand_k"]:<3s} route={m["route"]:<6s} '
          f'절감 {m["saved"]:6.1%} · 최저 보존율 {m["survival_worst"]:6.1%}')

## 6. 조건 비교

같은 코드에 조건만 바꿔 돌린 결과입니다. **숫자 하나가 아니라 표를 보세요.**
어떤 조건에서 무엇을 얻고 무엇을 잃는지가 이 랩의 결론입니다.

In [ ]:
table(
    ["설정", "코퍼스", "펼침", "라우팅", "다이제스트", "절감",
     "최저 보존율", "정답 절 적중"],
    [[n, m["dataset_name"], m["expand_k"], m["route"],
      f'{m["digest_chars"]}자' if m["digest_chars"] else "제목만",
      pct(m["saved"]), pct(m.get("survival_worst")),
      pct(m.get("router_hit_rate"))]
     for n, m, _ in results],
    align=["left", "left", "right", "left", "right", "right", "right", "right"],
    title="조건 비교",
    note="절감이 커도 최저 보존율이 0% 면 그 질문에는 답할 수 없습니다.",
)

by = {n: m for n, m, _ in results}
if "k1" in by and "k1-no-router" in by:
    a, b = by["k1"], by["k1-no-router"]
    print(f'같은 k=1, 비슷한 절감({a["saved"]:.1%} vs {b["saved"]:.1%}) 인데')
    print(f'최저 보존율은 {a["survival_worst"]:.1%} vs {b["survival_worst"]:.1%} 입니다.')
    print("차이는 압축 알고리즘이 아니라 무엇을 펼칠지 고르는 방법입니다.")
if "k1" in by and "k1-title-only" in by:
    d = by["k1-title-only"]["saved"] - by["k1"]["saved"]
    print(f'\n미리보기 24자를 지우면 절감이 {d:.1%}p 뜁니다 — '
          f'절 제목이 이미 설명적이면 미리보기는 순수 낭비입니다.')
if "k1-short" in by:
    print(f'\n짧은 산문(k1-short)은 절감 {by["k1-short"]["saved"]:.1%} — '
          f'핸들 표시와 색인 헤더가 원문만큼 큽니다.')

## 7. 절감률을 곧이곧대로 믿으면 안 되는 이유

**위 표의 절감률은 "꺼내기 전" 한 번의 컨텍스트만 잰 것입니다.**
실제로는 이렇게 흘러갑니다.

| 호출 | 보내는 것 |
|---|---|
| 1번째 | 다이제스트 (모델이 `[[b2]]` 를 달라고 함) |
| 2번째 | 다이제스트 + 꺼낸 블록 (이제 답함) |

**입력 토큰은 두 번 다 과금됩니다.** 다이제스트를 두 번 보내기 때문에,
한 번 묻고 끝이면 절감이 절반 이하로 줄어듭니다. 아래에서 실제로 계산합니다.

In [ ]:
n_doc = len(cases)
full = sum(counter(x.text) for x in cases) / n_doc          # 문서당 원문
digest = by["k0-digest-only"]["tokens_after"] / n_doc        # 문서당 다이제스트
expanded = by["k1"]["tokens_after"] / n_doc                  # 다이제스트 + 블록 1개

table(
    ["방식", "호출", "입력 토큰 합 (문서 1건 기준)", "절감"],
    [["원문을 통째로", "1회", f"{full:,.0f}", "기준"],
     ["핸들 — 위 표가 보여준 것", "1회분만", f"{expanded:,.0f}",
      pct(1 - expanded / full)],
     ["핸들 — 실제 (왕복 포함)", "2회", f"{digest + expanded:,.0f}",
      pct(1 - (digest + expanded) / full)]],
    align=["left", "right", "right", "right"],
    title="왕복을 세면 이야기가 달라집니다",
    note="다이제스트를 두 번 보내므로 절감이 크게 깎입니다. 지연도 2배입니다.",
)

그럼 언제 이깁니까. 세 가지 경우입니다.

1. **같은 문서에 질문을 여러 번** — 색인을 만드는 비용이 나눠집니다
2. **문서가 훨씬 클 때** — 다이제스트 비중이 작아집니다
3. **프롬프트 캐시가 먹을 때** — 매번 같은 다이제스트라 캐시 적중률이 높습니다
   (Azure 기준 최소 1,024토큰, 128토큰 단위)

1번을 계산해 봅니다. 색인은 한 번 만들고, 질문마다 블록만 새로 꺼냅니다.

In [ ]:
block = expanded - digest              # 블록 하나의 순수 비용

table(
    ["질문 수", "원문 매번", "핸들 (왕복 포함)", "절감"],
    [[q, f"{full * q:,.0f}",
      f"{digest * (q + 1) + block * q:,.0f}",
      pct(1 - (digest * (q + 1) + block * q) / (full * q))]
     for q in (1, 2, 3, 5, 10)],
    align=["right", "right", "right", "right"],
    title="같은 문서에 여러 번 물을 때",
    note="색인은 한 번, 블록은 매번. 질문이 늘수록 이득이 붙습니다.",
)

print(f"문서당 원문 {full:,.0f} · 다이제스트 {digest:,.0f} · 블록 {block:,.0f} 토큰")
print("이 코퍼스는 문서가 800자대로 짧은 편입니다.")
print("실제 장문(수천~수만 토큰)에서는 다이제스트 비중이 훨씬 작아 이득이 큽니다.")

## 9. 진짜로 줄었나 — API 응답으로 확인하기

여기까지의 절감률은 전부 **tiktoken 추정치**입니다. 실제로 청구되는 값은
API 응답의 `usage.input_tokens` 이고, 둘이 항상 같지는 않습니다.

| 왜 어긋나나 | 얼마나 |
|---|---|
| 메시지 포맷 오버헤드 (역할 구분자 등) | 텍스트당 상수 (실측 +6) |
| 배포 모델의 토크나이저가 tiktoken 과 다를 수 있음 | 모델마다 |
| 압축 결과의 특수 문자를 모델이 어떻게 쪼개는지 | **해봐야 압니다** |

마지막 줄이 중요합니다. 이 랩의 압축 결과에는 `[[b2]]` 같은 **핸들 표시**가 섞입니다. 대괄호가 몇 토큰으로 쪼개지는지는 실제로 불러봐야 압니다.

그래서 몇 건만 뽑아 **압축 전과 후를 각각 실제로 보내보고**, 응답이 알려주는
토큰 수로 절감률을 다시 계산합니다. 호출은 케이스당 2회이고 캐시됩니다.

In [ ]:
from kit import verify

DEPLOY = env.get("AZURE_OPENAI_DEPLOYMENT")
BILLED = None                     # 아래 리포트에서 다시 씁니다

try:
    kcfg = C.load('configs/k1.yaml')
    pairs = [(c.id, c.text,
              compress(c.text, question=c.question or '', **kcfg.params)[0])
             for c in cases]
    r = verify.billed(pairs, deployment=DEPLOY, model=DEPLOY, limit=3)
    BILLED = r["totals"]
    t = BILLED

    table(
        ["케이스", "tiktoken 전→후", "API 실측 전→후", "추정 절감", "실측 절감"],
        [[x["id"],
          f'{x["local_before"]:,} → {x["local_after"]:,}',
          f'{x["api_before"]:,} → {x["api_after"]:,}',
          pct(x["local_saved"]), pct(x["api_saved"])]
         for x in r["rows"]],
        foot=["합계",
              f'{t["local_before"]:,} → {t["local_after"]:,}',
              f'{t["api_before"]:,} → {t["api_after"]:,}',
              pct(t["local_saved"]), pct(t["api_saved"])],
        align=["left", "right", "right", "right", "right"],
        title=f'과금 기준으로 다시 재기 ({t["n"]}건)',
        note="'API 실측' 은 응답의 usage.input_tokens 를 그대로 읽은 값입니다.",
    )

    print(verify.verdict(t))
    print(f'텍스트당 오버헤드 {t["overhead_per_text"]:+.1f} 토큰 — '
          f'역할 구분자 같은 프레이밍이라 길이와 무관하게 붙습니다.')
    print(r["counter"].describe())
except Exception as e:
    print(f"과금 검증을 건너뜁니다 — {type(e).__name__}: {str(e)[:160]}")
    print("\\n자격증명이 있으면 아래로 준비하실 수 있습니다.")
    print("  cd labs && cp .env.example .env")
    print("없어도 위까지의 결과는 전부 유효합니다. 다만 추정치입니다.")

## 정리

- **정보를 버리는 게 아니라 밖에 둡니다** — 꺼내면 원문 그대로입니다
- **실패는 전부 라우팅 실패입니다** — 저장소는 아무것도 잃지 않습니다
- **토큰을 더 쓴다고 정확도가 오르지 않습니다** — 어디를 남기느냐가 정합니다
- **짧은 글에는 쓰면 안 됩니다** — 핸들 표시가 원문보다 커집니다
- **한 번 묻고 끝이면 손해입니다** — 같은 문서에 여러 번 물을 때 이깁니다

### 다음 랩

[`03-summarize-llm`](../03-summarize-llm/) — 밖에 두는 대신 실제로 버립니다.
유일하게 API 가 필요한 랩입니다.